In [1]:
# ============================================================
# [1단계] 필요한 도구(라이브러리) 설치하기
# ============================================================
# 요리를 하려면 칼, 도마, 냄비 같은 도구가 필요하죠?
# AI를 학습시키기 위해서도 이런 "도구"들이 필요한데,
# 이걸 컴퓨터에 설치하는 작업이에요.
#
# 아래 명령어 앞에 붙은 '!' 표시는
# "이건 컴퓨터(터미널)에게 직접 시키는 명령이야"라는 의미예요.
# ============================================================

# apt-get: 우분투(리눅스 운영체제)에서 프로그램을 설치하는 도구
# update: 설치 가능한 프로그램 목록을 최신 상태로 새로고침
!apt-get update

# ffmpeg: 영상/음성 파일을 다루는 만능 도구
#   -> 우리가 음성을 다룰 거니까 꼭 필요해요 (음성 파일 변환, 자르기 등)
# -y: 설치 중간에 "정말 설치할래?" 물어보면 자동으로 "예"라고 대답
!apt-get install -y ffmpeg

# pip: 파이썬(프로그래밍 언어) 전용 도구 설치 프로그램
# torchcodec: PyTorch(AI 학습에 쓰이는 핵심 도구)와 함께 영상/음성을 다룸
# ==0.3.0: 정확히 0.3.0 버전을 설치 (버전이 다르면 호환 안 될 수 있어서 콕 집어줌)
!pip install torchcodec==0.3.0

# datasets: 학습용 데이터를 쉽게 다운로드하고 관리해주는 도구
!pip install datasets==3.6.0

# 한 줄로 여러 도구를 한 번에 설치 (공백으로 구분)
# - librosa, soundfile: 음성 파일을 읽고 분석하는 도구
# - jiwer: 음성 인식 결과가 얼마나 정확한지 측정 (단어 오류율 계산)
# - evaluate: AI 성능을 평가하는 다양한 측정 도구
# - huggingface_hub: AI 모델을 인터넷에서 다운받는 곳(허깅페이스)과 연결
# - peft: 거대한 AI를 적은 자원으로 효율적으로 학습시키는 비법 도구
# - Levenshtein: 두 글자가 얼마나 다른지 계산 (오타 검출 같은 곳에 쓰임)
!pip install librosa soundfile jiwer evaluate huggingface_hub peft Levenshtein


Reading package lists... Done
E: Could not open lock file /var/lib/apt/lists/lock - open (13: Permission denied)
E: Unable to lock directory /var/lib/apt/lists/
W: Problem unlinking the file /var/cache/apt/pkgcache.bin - RemoveCaches (13: Permission denied)
W: Problem unlinking the file /var/cache/apt/srcpkgcache.bin - RemoveCaches (13: Permission denied)
E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?


In [2]:
# ============================================================
# [1.5단계] 사용할 GPU 지정하기
# ============================================================
# 컴퓨터에 GPU가 여러 개 있을 때 (0번, 1번, 2번 ... 처럼 번호가 붙음),
# "나는 이번 학습에 4번 GPU만 쓸게!"라고 미리 알려주는 작업이에요.
#
# 왜 미리 해야 할까요?
#   - 다른 사람과 GPU를 나눠 쓸 때 충돌 방지
#   - 특정 GPU만 사용해서 메모리/자원 격리
#   - 여러 GPU 중 비어있는 것만 골라 쓸 수 있음
#
# [중요] 이 설정은 반드시 'import torch' 보다 먼저 와야 해요!
#   - PyTorch가 GPU를 한 번 인식한 뒤에는 이 설정이 무시돼요
#   - 그래서 노트북에서 가장 위쪽(import 셀 직전)에 두는 거예요
# ============================================================

# os: 운영체제와 대화하는 도구 (환경변수 설정 등)
import os

# 환경변수 CUDA_VISIBLE_DEVICES를 "4"로 설정
# → PyTorch는 이제 4번 GPU만 보이게 됨 (나머지는 없는 셈 침)
# → 코드 안에서 'cuda:0' 이라고 쓰면 실제로는 4번 GPU를 가리킴
#    (눈에 보이는 GPU가 1개뿐이라 그게 자동으로 0번이 됨)
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])
print("이제 4번 GPU만 사용합니다.")


CUDA_VISIBLE_DEVICES = 4
이제 4번 GPU만 사용합니다.


In [3]:
# ============================================================
# [2단계] 설치한 도구들을 "꺼내서 쓸 준비" 하기
# ============================================================
# 1단계에서 도구를 컴퓨터에 "설치"했다면,
# 2단계에서는 그 도구를 실제로 "지금부터 쓸게요!" 하고 불러오는 거예요.
# 마치 서랍에 있던 연필을 책상 위에 꺼내놓는 것과 비슷해요.
#
# 'import A' = A라는 도구를 가져와서 쓸게요
# 'from A import B' = A라는 도구 상자에서 B라는 부품만 꺼내 쓸게요
# ============================================================

import torch              # PyTorch: AI 학습의 핵심 엔진 (계산을 GPU로 빠르게 해줌)
import random             # 무작위 숫자를 만드는 도구 (예: 주사위 굴리기)
from dataclasses import dataclass    # 데이터를 깔끔하게 정리하는 상자 만드는 도구
from typing import Any, Dict, List, Union  # 데이터의 "종류"를 명시할 때 쓰는 도구 (코드 가독성용)

# 허깅페이스(AI 모델 공유 사이트) 로그인 도구
# - 비공개 데이터셋이나 모델을 받을 때 필요해요
from huggingface_hub import login

# datasets: 학습용 데이터를 다운로드하고 관리하는 도구
# - load_dataset: 데이터셋을 인터넷에서 다운로드
# - Audio: 음성 데이터를 다루기 위한 형식 지정
from datasets import load_dataset, Audio

# peft: 거대한 AI를 효율적으로 학습시키는 비법 도구 모음
# - prepare_model_for_kbit_training: 모델을 가볍게 학습할 수 있도록 준비
# - LoRA: 모델 전체가 아닌 "작은 일부"만 학습하는 방식 (메모리 절약 핵심!)
# - get_peft_model: 일반 모델을 LoRA 학습용 모델로 변환
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# transformers: 허깅페이스가 만든 AI 모델 도구 (오늘의 주인공!)
# Whisper는 OpenAI가 만든 "음성 → 글자" 변환 AI 모델이에요
from transformers import (
    WhisperFeatureExtractor,            # 음성을 AI가 이해할 수 있는 숫자로 변환
    WhisperTokenizer,                   # 글자를 AI가 이해할 수 있는 숫자(토큰)로 변환
    WhisperProcessor,                   # 위 두 가지를 한 번에 처리해주는 종합 도구
    WhisperForConditionalGeneration,    # Whisper 모델 본체 (실제 음성→글자 변환)
    Seq2SeqTrainingArguments,           # 학습 설정값 (학습 속도, 횟수 등)
    Seq2SeqTrainer                      # 실제로 학습을 진행시키는 "코치" 역할
)

# 모델의 성능을 평가하기 위한 도구 (시험 채점기 같은 역할)
import evaluate


In [4]:
# ============================================================
# [3단계] 랜덤 시드 설정 - "재현 가능한 실험"을 위한 작업
# ============================================================
# 컴퓨터에서 "무작위(랜덤)"이라고 해도 사실은 계산식으로 만든 가짜 무작위예요.
# 그래서 "시작 숫자"를 똑같이 정해두면, 매번 같은 무작위 결과가 나와요.
#
# 왜 이게 중요할까요?
# - 오늘 학습한 결과와 내일 학습한 결과가 똑같이 나와야
#   "왜 다를까?" 헷갈리지 않고 실험을 비교할 수 있어요.
# - 마치 주사위 굴리기를 녹화한 영상을 다시 트는 것처럼,
#   똑같은 결과가 나오도록 해주는 거예요.
# ============================================================

# 시작 숫자(시드)는 아무 숫자나 정해도 돼요. 여기서는 8282를 골랐어요.
# (개발자들은 종종 자기가 좋아하는 숫자를 시드로 써요)
RANDOM_SEED = 8282

# 파이썬 기본 랜덤 도구에 시드를 설정
random.seed(RANDOM_SEED)

# PyTorch(AI 학습 도구)에도 같은 시드를 설정
torch.manual_seed(RANDOM_SEED)

# GPU(그래픽카드)를 사용할 수 있다면 GPU 쪽 랜덤도 고정
# - cuda: NVIDIA 그래픽카드용 빠른 계산 기술
# - is_available(): GPU가 컴퓨터에 있는지 확인
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)  # 모든 GPU에 시드 설정

# 설정이 잘 됐는지 화면에 출력해서 확인
# f"..." : 문자열 안에 {변수}를 끼워넣는 표기법
print(f"랜덤 시드가 {RANDOM_SEED}로 설정되었습니다.")


랜덤 시드가 8282로 설정되었습니다.


In [5]:
# ============================================================
# [4단계] 학습용 데이터셋 다운로드하기
# ============================================================
# AI를 학습시키려면 "교과서" 역할을 하는 데이터가 필요해요.
# 음성 인식 AI라면 [음성 파일 + 그 음성이 무슨 말인지 적힌 텍스트]
# 이렇게 짝지어진 데이터가 필요해요.
#
# 여기서는 "한국어 주소 음성 데이터셋"을 사용해요.
# 예: 음성 "서울특별시 강남구 테헤란로..." + 텍스트 "서울특별시 강남구 테헤란로..."
# ============================================================

# load_dataset: 허깅페이스(데이터셋 공유 사이트)에서 데이터를 받아오는 함수
# "daje/korean-address-voice-v2"
#   - daje: 데이터를 올린 사람(또는 단체) 이름
#   - korean-address-voice-v2: 데이터셋 이름 (한국어 주소 음성 v2)
datasets = load_dataset("daje/korean-address-voice-v2")

# 다운받은 데이터의 구조를 화면에 출력해서 확인
# 출력 예시:
#   - train: 학습용 데이터 3,400개 (시험 공부할 때 보는 문제집)
#   - test: 시험용 데이터 340개 (실제 시험 문제)
#   - features: 'audio'(음성), 'text'(정답 글자) 두 가지로 구성
print(datasets)


DatasetDict({
    train: Dataset({
        features: ['audio', 'text'],
        num_rows: 3400
    })
    test: Dataset({
        features: ['audio', 'text'],
        num_rows: 340
    })
})


In [6]:
# ============================================================
# [5단계] Whisper 음성 인식 AI 모델 가져오기
# ============================================================
# Whisper는 OpenAI(챗GPT를 만든 회사)가 만든 음성 → 글자 변환 AI예요.
# 이미 엄청난 양의 음성 데이터로 학습이 끝나있는 "똑똑한 AI"인데,
# 우리는 이 AI를 가져와서 "한국어 주소"를 더 잘 알아듣게 살짝 더 가르칠 거예요.
# (이걸 "파인튜닝(Fine-tuning)"이라고 해요)
# ============================================================

# 사용할 모델 이름을 변수에 저장 (나중에 여러 번 쓰기 편하라고)
# "openai/whisper-large-v3-turbo":
#   - openai: 모델을 만든 회사
#   - whisper-large-v3-turbo: 모델 이름 (large 크기, v3 버전의 빠른 turbo 모델)
MODEL_NAME = "openai/whisper-large-v3-turbo"

# ① 음성 → 숫자 변환기 (Feature Extractor)
# 컴퓨터는 음성을 직접 이해할 수 없으니, 숫자(특징값)로 바꿔서 줘야 해요.
# .from_pretrained(이름): 인터넷에서 이미 학습된 도구를 다운받음
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)

# ② 글자 → 숫자 변환기 (Tokenizer)
# 마찬가지로 글자도 숫자로 바꿔야 AI가 알아들어요.
# 예: "안녕" → [4523, 891] 같은 숫자 코드로 변환
# - language="Korean": 한국어로 처리해달라고 명시
# - task="transcribe": 음성을 텍스트로 받아쓰기 모드 (translate=번역과 구분)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, language="Korean", task="transcribe")

# ③ 위의 두 가지를 한꺼번에 처리해주는 종합 도구
# 학습할 때는 이 processor를 주로 써요 (한 번에 음성+글자 처리 가능)
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="Korean", task="transcribe")


In [7]:
# ============================================================
# [6단계] 데이터셋 속 텍스트 한 개 미리 들여다보기
# ============================================================
# 데이터가 어떻게 생겼는지 한 개만 꺼내봐요 (실제 모양 확인용)
# ============================================================

# datasets["train"]: 학습용 데이터 묶음에서
# [0]: 0번째(첫 번째) 데이터를 꺼내서
# ["text"]: 그 중 'text'(정답 글자) 부분만 가져옴
# (참고: 컴퓨터는 0부터 세요. 첫 번째 = 0번)
input_str = datasets["train"][0]["text"]

# 변수 이름만 적으면 그 안의 값을 화면에 보여줘요 (주피터 노트북의 편리한 기능)
# 결과: '서울특별시영등포구압구정로136 SK뷰293동3047호' 같은 주소 텍스트
input_str


'서울특별시영등포구압구정로136 SK뷰293동3047호'

In [8]:
# ============================================================
# [7단계] 토크나이저(글자 → 숫자 변환기) 작동 확인하기
# ============================================================
# 토크나이저가 글자를 숫자로 바꾸고, 다시 글자로 되돌릴 때
# 원본과 똑같이 나오는지 확인하는 단계예요.
# (왕복 여행 후 짐이 빠지지 않았는지 검사하는 것과 비슷)
# ============================================================

# 글자를 숫자(토큰 ID)로 변환
# .input_ids: 변환 결과 중 "숫자 코드 목록"만 꺼내기
# 예: "안녕" → [50264, 50360, 50364, 4523, 891, 50257] 같은 숫자 배열
labels = tokenizer(input_str).input_ids

# 숫자를 다시 글자로 되돌리기 (디코딩)
# skip_special_tokens=False: AI가 내부적으로 쓰는 특수 표시도 모두 포함
# 예: <|startoftranscript|>(시작 표시), <|ko|>(한국어 표시), <|endoftext|>(끝 표시) 등
decoded_with_special = tokenizer.decode(labels, skip_special_tokens=False)

# 숫자를 다시 글자로 되돌리기 (특수 표시는 제외)
# skip_special_tokens=True: 사람이 보기 좋게 특수 표시 빼고 순수 글자만
decoded_str = tokenizer.decode(labels, skip_special_tokens=True)

# 비교 결과를 화면에 출력
# (스페이스 개수를 맞춰서 표 형태로 보기 좋게 정렬)
print(f"입력:                {input_str}")              # 원본
print(f"특수토큰 포함 디코딩:    {decoded_with_special}")    # AI 내부 표시까지
print(f"특수토큰 제외 디코딩:    {decoded_str}")            # 사람용 (순수 글자만)

# 원본과 변환 후 결과가 같은지 비교 (True면 OK, False면 문제 있음)
# ==: "두 값이 같은가?" 비교 연산자
print(f"원본과 동일 여부:       {input_str == decoded_str}")


입력:                서울특별시영등포구압구정로136 SK뷰293동3047호
특수토큰 포함 디코딩:    <|startoftranscript|><|ko|><|transcribe|><|notimestamps|>서울특별시영등포구압구정로136 SK뷰293동3047호<|endoftext|>
특수토큰 제외 디코딩:    서울특별시영등포구압구정로136 SK뷰293동3047호
원본과 동일 여부:       True


In [9]:
# ============================================================
# [8단계] 데이터 한 개의 전체 모습 들여다보기
# ============================================================
# 6단계에서는 텍스트만 봤다면, 이번엔 음성+텍스트 통째로 확인해요.
# ============================================================

# 학습 데이터의 첫 번째 항목 전체를 출력
# 출력 결과 살펴보면 이런 구조예요:
# {
#   'audio': {
#       'path': '파일이름.mp3',           # 음성 파일 이름
#       'array': [0.0, 0.0, ..., 0.0],   # 음성을 숫자로 표현한 긴 배열
#                                          (소리 파형을 점점이 찍은 숫자들)
#       'sampling_rate': 24000             # 1초당 24,000개의 숫자로 표현 (24kHz)
#   },
#   'text': '서울특별시영등포구...'         # 음성에 해당하는 정답 텍스트
# }
#
# [참고] sampling_rate(샘플링 레이트):
#    1초 음성을 몇 개의 숫자로 쪼개 표현하는지를 나타내요.
#    숫자가 클수록 음질이 좋아요 (대신 데이터 용량은 커짐).
print(datasets["train"][0])


{'audio': {'path': '서울특별시영등포구압구정로136_SK뷰293동3047호.mp3', 'array': array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
       2.81896450e-06, 2.82983729e-06, 8.46276862e-08]), 'sampling_rate': 24000}, 'text': '서울특별시영등포구압구정로136 SK뷰293동3047호'}


In [10]:
# ============================================================
# [9단계] 데이터 전처리 함수 정의 (첫 번째 버전 - 참고용)
# ============================================================
# AI에게 데이터를 먹이기 전에 "AI가 잘 소화할 수 있는 형태"로
# 바꿔주는 작업이 필요해요. 이걸 "전처리(preprocessing)"라고 해요.
#
# [주의] 이 셀은 "원래 방식"의 예시이고,
#    아래 다음 셀에서 더 올바른 방법으로 다시 정의할 거예요.
#    (샘플링 레이트 문제 때문에 - 다음 셀 설명 참고)
# ============================================================

# def: 함수(자주 쓰는 작업을 묶어둔 것)를 만들 때 쓰는 키워드
# prepare_dataset이라는 이름으로, batch(데이터 묶음)를 받는 함수를 만들어요
def prepare_dataset(batch):
    # 오디오 데이터를 로드하고 48kHz에서 16kHz로 리샘플링
    # (실제로는 24kHz → 16kHz이지만 주석 원본은 48kHz로 적혀있음)
    audio = batch["audio"]  # batch에서 음성 정보만 꺼냄

    # 음성의 숫자 배열을 AI가 이해하는 "log-Mel 스펙트로그램"으로 변환
    # [참고] log-Mel 스펙트로그램:
    #    음성을 "주파수의 그림"으로 변환한 것.
    #    AI 음성 인식의 표준 입력 형식이에요.
    # [0]은 결과 배열의 첫 번째(유일한) 항목을 꺼내는 거예요
    batch["input_features"] = feature_extractor(
        audio["array"],                         # 음성 숫자 배열
        sampling_rate=audio["sampling_rate"]    # 샘플링 레이트 (이게 문제!)
    ).input_features[0]

    # 정답 텍스트를 토큰(숫자 코드)으로 변환해서 'labels' 칸에 저장
    # 학습 시 AI는 이 labels를 "정답"으로 보고 자기 답안과 비교해요
    batch["labels"] = tokenizer(batch["text"]).input_ids

    # 처리된 batch를 반환(돌려줌)
    return batch


In [11]:
# ============================================================
# [10단계] 샘플링 레이트 맞추기 + 전체 데이터 전처리 (실제 작동본!)
# ============================================================
# [핵심 문제]
#   Whisper AI는 "1초당 16,000개 숫자(16kHz)" 음성으로 학습됐어요.
#   그런데 우리 데이터는 "1초당 24,000개 숫자(24kHz)"로 되어있어요.
#   → 비율이 안 맞으면 AI가 음성을 "느린/빠른 소리"로 잘못 인식해요!
#
# 해결책: 우리 음성을 16kHz로 "리샘플링" (재변환)
# 마치 60프레임 영상을 30프레임으로 바꾸는 것과 비슷해요.
# ============================================================

# 문제 원인 정리 (코드에 영향 없는 설명용 주석)
# Whisper 모델: 16,000 Hz (16kHz) 샘플링 레이트로 학습됨
# 현재 오디오 데이터: 24,000 Hz (24kHz)로 되어 있음
# Google TTS로 생성한 오디오가 24kHz로 되어 있어서, Whisper가 요구하는 16kHz와 맞지 않습니다.

# .cast_column: 특정 컬럼(여기서는 'audio')의 형식을 바꾸는 명령
# Audio(sampling_rate=16000): "이제부터 이 음성은 16kHz로 다뤄줘"
# → 데이터를 꺼낼 때 자동으로 16kHz로 변환되어 나옴
datasets = datasets.cast_column("audio", Audio(sampling_rate=16000))

# 전처리 함수를 다시 정의 (이번엔 16kHz로 명확히 지정)
def prepare_dataset(batch):
    audio = batch["audio"]
    # 이미 16kHz로 리샘플링되어 있음 (위에서 cast_column으로 변환했으니까)
    batch["input_features"] = feature_extractor(
        audio["array"],
        sampling_rate=16000   # 명시적으로 16000 사용 (안전하게 콕 집어줌)
    ).input_features[0]
    # 정답 텍스트를 토큰 숫자로 변환
    batch["labels"] = tokenizer(batch["text"]).input_ids
    return batch

# 위에서 만든 함수를 모든 데이터(3,400개 + 340개)에 적용!
# .map(): 데이터 하나하나에 함수를 적용하는 명령
#   (마치 학생들에게 같은 시험지를 한 명씩 채점하는 것)
# - remove_columns: 원본 'audio', 'text' 컬럼은 이제 필요 없으니 삭제
#   (변환 결과인 'input_features', 'labels'만 남김 → 용량 절약)
# - num_proc=16: 동시에 16개의 작업자(프로세스)가 병렬 처리 → 16배 빠름!
datasets = datasets.map(
    prepare_dataset,
    remove_columns=datasets.column_names["train"],
    num_proc=16
)


In [12]:
# ============================================================
# [11단계] 모델 불러오기 + LoRA로 효율적 학습 설정
# ============================================================
# Whisper Large v3 Turbo 모델은 8억 개가 넘는 "신경망 부품(파라미터)"을
# 가진 거대한 AI예요. 이걸 전부 새로 학습시키려면
# 어마어마한 GPU 메모리와 시간이 필요해요.
#
# [핵심] LoRA(Low-Rank Adaptation, "로라"):
#    전체 모델을 학습하는 대신, 작은 "외부 부속품"만 추가로 학습하는 기술.
#    원본 모델은 그대로 두고, 옆에 작은 어댑터만 끼우는 느낌이에요.
#    → 학습할 파라미터가 1.6%로 줄어 들어도 성능은 비슷!
#    → 일반 게이밍 PC로도 거대 모델 학습 가능!
# ============================================================

print("모델 로드 중...")  # 시간이 좀 걸리니까 진행 상황을 알려줘요

# Whisper 모델 본체를 인터넷에서 다운받아 메모리에 올림
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,                # 5단계에서 정한 모델 이름
    device_map="auto",         # GPU/CPU에 자동으로 분배해서 올림
    use_cache=False,           # 캐시 끄기 (gradient checkpointing과 호환되게)
                               # ※ gradient checkpointing: 메모리 절약 기법
)

# 한국어 음성 인식 모드로 설정 (영어/일본어 등이 아닌 한국어로 받아쓰기)
model.generation_config.language = "ko"        # 출력 언어 = 한국어
model.generation_config.task = "transcribe"    # 작업 = 받아쓰기(번역 X)
model.generation_config.forced_decoder_ids = None  # 강제 토큰 설정 해제

# ───── 2단계: LoRA 설정 ─────
# LoRA를 어떻게 적용할지 세부 옵션을 정해요
lora_config = LoraConfig(
    r=32,             # rank(랭크) = 어댑터의 "크기". 클수록 표현력↑ 메모리↑
    lora_alpha=64,    # 학습 강도 조절 (보통 r의 2배로 설정)
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
                      # ↑ 모델 안의 어떤 부품에 LoRA를 끼울지 (어텐션 부분에 적용)
    lora_dropout=0.05,  # 5% 확률로 학습 중 일부를 무작위로 꺼서 과적합 방지
                        # (학생이 답을 외우지 못하게 가끔 시험문제 바꾸는 느낌)
    bias="none"       # 편향(bias) 파라미터는 학습 안 함
)

# ───── 3단계: 일반 모델 → LoRA 모델로 변환 ─────
# get_peft_model: 모델에 LoRA 어댑터를 부착해줌
model = get_peft_model(model, lora_config)

# ───── 4단계: 학습 가능한 파라미터 확인 ─────
# 출력 예: "trainable params: 13M || all params: 821M || trainable%: 1.59%"
# → 전체의 1.59%만 학습! 나머지 98.41%는 그대로 고정.
model.print_trainable_parameters()

# gradient checkpointing(메모리 절약 기법)과 LoRA가 충돌할 수 있어서
# 캐시를 다시 한 번 명시적으로 꺼줌 (안전장치)
model.config.use_cache = False


모델 로드 중...
trainable params: 13,107,200 || all params: 821,985,280 || trainable%: 1.5946


In [13]:
# ============================================================
# [12단계] 데이터 콜레이터(Data Collator) 만들기
# ============================================================
# [데이터 콜레이터란?]
#   여러 개의 데이터를 한 번에 묶어 AI에게 "한 입 크기로" 만들어주는 도구.
#   (콜레이터 = collate = 정리하다)
#
# 왜 필요할까요?
#   - AI는 한 번에 여러 데이터를 동시에 학습해요 (이걸 batch라고 함)
#   - 그런데 각 데이터의 "길이"가 다 달라요
#     (예: "서울시 강남구" 짧음 / "서울특별시 영등포구 압구정로..." 김)
#   - 길이가 다르면 한 묶음으로 못 만들어요 → 짧은 건 빈칸으로 채워야 함
#   - 이 빈칸 채우기(=패딩) 작업을 하는 게 콜레이터예요!
# ============================================================

# @dataclass: 데이터 보관용 클래스를 쉽게 만들어주는 마법
#   (생성자 코드를 자동으로 만들어줘서 편리해요)
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    # 이 콜레이터가 가지고 있을 도구들 선언
    processor: Any                  # 5단계에서 만든 종합 처리기
    decoder_start_token_id: int     # "이제부터 글자를 출력 시작!" 표시 토큰

    # __call__: 객체를 함수처럼 호출했을 때 실행되는 특별한 메서드
    # 즉, collator(데이터) 이렇게 쓸 수 있게 됨
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # 입력(음성)과 레이블(텍스트)은 길이도 다르고 패딩 방식도 다르므로
        # 따로따로 처리해요.

        # ───── 1) 음성 데이터 패딩 ─────
        # 리스트 컴프리헨션: [f(x) for x in 리스트] 형태로 새 리스트 만드는 문법
        # 각 데이터에서 'input_features'(음성)만 꺼내 모음
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        # .pad(): 짧은 음성 뒤에 빈칸을 채워서 가장 긴 음성과 길이 맞춤
        # return_tensors="pt": 결과를 PyTorch 텐서(다차원 배열) 형식으로 받음
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # ───── 2) 텍스트(레이블) 데이터 패딩 ─────
        # 각 데이터에서 'labels'(정답 토큰)만 꺼내 모음
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # 마찬가지로 짧은 텍스트도 가장 긴 텍스트에 맞춰 패딩
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # [중요] -100이라는 특별한 값으로 패딩 자리 표시
        # AI 학습 시 "이 위치는 채워넣은 빈칸이니까 점수 계산에서 빼!"라는 약속된 신호
        # masked_fill: attention_mask가 1이 아닌(=패딩) 위치를 -100으로 채움
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # 모든 데이터 맨 앞에 시작 토큰이 붙어있다면 그건 제거
        # (어차피 학습 과정에서 자동으로 다시 붙음 → 중복 방지)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]   # [:, 1:] = 첫 번째 토큰 빼고 나머지

        # batch에 labels 추가
        batch["labels"] = labels

        # 완성된 batch 반환 (이게 AI에게 한 입 떠먹여 줄 데이터)
        return batch


In [14]:
# ============================================================
# [13단계] 데이터 콜레이터 "실제로 만들어 사용 준비" 하기
# ============================================================
# 12단계에서는 콜레이터의 "설계도(클래스)"만 만들었어요.
# 여기서는 그 설계도로 실제 콜레이터 "인스턴스(객체)"를 만들어요.
# (설계도 = 붕어빵 틀 / 인스턴스 = 실제 붕어빵)
# ============================================================

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,    # 5단계에서 만든 종합 처리기 전달
    # decoder_start_token_id: 디코더(글자 만드는 부분)의 시작 신호 토큰 ID
    # model.config에 모델의 다양한 설정값이 저장돼 있어요
    decoder_start_token_id=model.config.decoder_start_token_id,
)


In [15]:
# ============================================================
# [14단계] 콜레이터가 잘 작동하는지 작은 테스트 해보기
# ============================================================
# 본격 학습 전에 "데이터를 묶어주는 콜레이터"가 제대로 작동하는지
# 샘플 2개로 미리 확인해보는 단계예요.
# (요리 전에 간을 보는 느낌)
# ============================================================

# 테스트용 샘플 2개 골라서 리스트로 만들기
sample_features = [
    datasets["train"][0],  # 첫 번째 샘플 (0번)
    datasets["train"][1],  # 두 번째 샘플 (1번)
]

# 콜레이터에 2개 샘플을 넣어 배치(묶음)로 만들기
# data_collator(...)는 위에서 만든 __call__이 실행되는 것
batch = data_collator(sample_features)

# ───── 결과 확인 ─────

print("=== 배치 구조 ===")

# .shape: 텐서(다차원 배열)의 모양을 보여줌
# input_features: 음성 데이터
#   예: [2, 128, 3000]
#       → 2개 샘플, 128개 주파수 채널, 3000개 시간 프레임
print(f"input_features 크기: {batch['input_features'].shape}")

# labels: 정답 텍스트(토큰 숫자)
#   예: [2, 30] → 2개 샘플, 각 30개 토큰 길이
print(f"labels 크기: {batch['labels'].shape}")

# 실제 토큰 숫자들 들여다보기
print("\n=== 첫 번째 샘플의 레이블 (처음 20개 토큰) ===")
# [0]: 첫 번째 샘플, [:20]: 처음 20개 토큰만 슬라이싱
print(batch['labels'][0][:20])

print("\n=== 두 번째 샘플의 레이블 (처음 20개 토큰) ===")
print(batch['labels'][1][:20])

# 패딩(빈칸 채우기) 확인
# 짧은 텍스트 뒤에는 -100이 채워져 있어야 정상
print("\n=== 패딩 확인 (레이블의 마지막 10개 토큰) ===")
# [-10:]: 끝에서 10개 토큰 (음수 인덱스는 "뒤에서부터" 의미)
print(f"첫 번째 샘플: {batch['labels'][0][-10:]}")
print(f"두 번째 샘플: {batch['labels'][1][-10:]}")  # -100이 보이면 패딩 잘 된 것


=== 배치 구조 ===
input_features 크기: torch.Size([2, 128, 3000])
labels 크기: torch.Size([2, 30])

=== 첫 번째 샘플의 레이블 (처음 20개 토큰) ===
tensor([50264, 50360, 50364,  2393, 15580,  5963,   117, 37604,  3833, 11958,
        36912, 30600,  7675,  1457,   243,  7675,  6170, 12888,  7668,    21])

=== 두 번째 샘플의 레이블 (처음 20개 토큰) ===
tensor([50264, 50360, 50364,  2393, 15580,  5963,   117, 37604,  3833, 36074,
         8097, 11545,  9520,  1831,  6826, 16270,  5254, 15390, 12504,  4264])

=== 패딩 확인 (레이블의 마지막 10개 토큰) ===
첫 번째 샘플: tensor([21483,   167, 33067, 11871,    18, 23056,  3446, 14060, 14705, 50257])
두 번째 샘플: tensor([ 1129, 22452,    17, 23056,    19,  5211,    24, 14705, 50257,  -100])


In [16]:
# ============================================================
# [15단계] 평가 메트릭(성능 측정 지표) 만들기
# ============================================================
# AI가 학습되는 동안 "잘 배우고 있나?"를 측정해야 해요.
# 마치 시험 점수를 매기는 것처럼요.
#
# 음성 인식에서 자주 쓰는 두 가지 점수:
# 1) WER (Word Error Rate): 단어 단위 오류율
#    예: "강남구 역삼동" → "강남구 역삽동" → 단어 1개 틀림 → WER 50%
#
# 2) CER (Character Error Rate): 글자 단위 오류율 (한국어에 더 좋음)
#    예: "강남구" → "강난구" → 글자 1개 틀림 → CER 약 33%
#    한국어는 띄어쓰기 기준이 애매해서 WER보다 CER이 더 정확함
#
# 점수는 낮을수록 좋아요! (오류율이니까)
# ============================================================

# 평가 도구 두 가지 로드
wer_metric = evaluate.load("wer")  # 단어 오류율 계산기
cer_metric = evaluate.load("cer")  # 글자 오류율 계산기

# 학습 중 자동으로 호출될 평가 함수 정의
# pred: 모델이 예측한 결과 객체
def compute_metrics(pred):
    pred_ids = pred.predictions   # AI가 예측한 토큰 숫자들
    label_ids = pred.label_ids    # 정답 토큰 숫자들

    # 패딩 자리(-100)를 진짜 pad_token으로 복원
    # (디코딩(숫자→글자)할 때 -100은 처리가 안 되니까)
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # 토큰 숫자 → 사람이 읽을 수 있는 글자로 변환
    # batch_decode: 여러 개를 한꺼번에 디코딩
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)   # AI 답안
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True) # 정답

    # WER 계산 (단위를 %로 만들려고 100 곱함)
    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)

    # CER 계산 (한국어에 특히 유용)
    cer = 100 * cer_metric.compute(predictions=pred_str, references=label_str)

    # 한국어 전용 글자 정확도 (다음 셀에서 정의할 함수 호출)
    char_acc = calculate_korean_char_accuracy(pred_str, label_str)

    # 결과를 딕셔너리(이름표 붙은 사전)로 반환
    # 학습 로그에 이 이름들로 점수가 표시됨
    return {
        "wer": wer,                    # 단어 오류율 (낮을수록 좋음)
        "cer": cer,                    # 글자 오류율 (낮을수록 좋음)
        "char_accuracy": char_acc      # 글자 정확도 (높을수록 좋음)
    }


In [17]:
# ============================================================
# [16단계] 한국어 글자 정확도 계산 함수 만들기
# ============================================================
# 15단계의 compute_metrics에서 호출하는 보조 함수예요.
# "한국어 띄어쓰기는 좀 헷갈리니, 띄어쓰기는 무시하고
#  글자만 보고 정확도를 계산하자"는 아이디어예요.
# ============================================================

def calculate_korean_char_accuracy(predictions, references):
    """한국어 문자 단위 정확도 계산"""
    # ↑ 큰따옴표 3개로 둘러싸인 부분은 함수 설명(docstring)이에요

    # 통계용 변수 초기화
    total_chars = 0     # 정답 글자 총 개수
    correct_chars = 0   # 맞춘 글자 개수

    # zip(): 두 리스트를 짝지어 동시에 순회
    # predictions와 references를 한 쌍씩 묶어서 비교
    for pred, ref in zip(predictions, references):
        # 띄어쓰기(공백) 제거 후 글자 하나씩 리스트로 만듦
        # 예: "강 남구" → "강남구" → ['강', '남', '구']
        pred_chars = list(pred.replace(" ", ""))   # AI 답안 글자 목록
        ref_chars = list(ref.replace(" ", ""))     # 정답 글자 목록

        # 두 리스트 중 짧은 쪽 길이까지만 비교
        # (둘 길이가 다를 수 있으니 안전하게)
        min_len = min(len(pred_chars), len(ref_chars))

        # range(N): 0, 1, 2, ..., N-1 까지 숫자를 차례로 생성
        for i in range(min_len):
            # i번째 글자가 같으면 맞춘 개수 +1
            if i < len(pred_chars) and pred_chars[i] == ref_chars[i]:
                correct_chars += 1   # += 1은 "기존 값에 1을 더해라"는 의미

        # 정답 글자 수 누적
        total_chars += len(ref_chars)

    # 정확도 계산 (백분율)
    # 삼항 연산자: 값1 if 조건 else 값2
    #   → 조건이 참이면 값1, 거짓이면 값2
    # total_chars가 0이면 나눗셈 오류 나니까 안전장치
    return (correct_chars / total_chars * 100) if total_chars > 0 else 0


In [18]:
# ============================================================
# [17단계] 학습 결과를 저장할 폴더 위치 지정
# ============================================================
# 학습 중간중간 모델을 저장(체크포인트)할 곳을 정해요.
# 학습이 도중에 멈춰도 다시 이어서 할 수 있고,
# 최종 모델도 여기에 저장돼요.
#
# "./model-v2":
#   . = 현재 폴더
#   / = 폴더 구분자
#   model-v2 = 만들어질 폴더 이름 (자동 생성됨)
# 즉, "현재 위치 아래에 model-v2라는 폴더에 저장해줘"라는 뜻
# ============================================================

OUTPUT_DIR = "./model-v2"


In [19]:
# ============================================================
# [18단계] 학습 옵션 설정 (가장 중요한 단계!)
# ============================================================
# AI를 어떻게 학습시킬지 세부 옵션을 정해요.
# 운동 계획표 짜는 것과 비슷해요:
#   - 며칠 동안? 하루에 몇 번?
#   - 강도는? 휴식은 언제?
#   - 어떤 기준으로 잘했는지 판단할지?
# ============================================================

training_args = Seq2SeqTrainingArguments(
    # ─────── 저장 관련 ───────
    output_dir=OUTPUT_DIR,                  # 17단계에서 정한 저장 위치

    # ─────── 배치 사이즈 (한 번에 몇 개 학습할지) ───────
    per_device_train_batch_size=32,         # GPU 1개당 학습 시 32개 묶음으로
    gradient_accumulation_steps=2,          # 32개씩 2번 모아서 → 사실상 64개로 학습
                                            # (메모리 부족할 때 쓰는 트릭)

    # ─────── 학습 속도/강도 ───────
    learning_rate=1e-4,                     # 학습률 = 0.0001 (1e-4 = 1 × 10⁻⁴)
                                            # AI가 한 번에 얼마나 크게 배울지
                                            # 너무 크면 휘청, 너무 작으면 안 배움
    warmup_ratio=0.1,                       # 처음 10%는 천천히 워밍업 (안정성↑)
    max_steps=100,                          # 총 학습 횟수 (100번만 학습)
                                            # 실제 운영에선 보통 더 많이 함
    fp16=True,                              # 16비트 부동소수점 사용 (메모리/속도 2배 절약)

    # ─────── 평가(시험) 관련 ───────
    per_device_eval_batch_size=64,          # 평가 시 한 번에 64개씩 처리
    eval_strategy="steps",                  # 일정 step마다 평가
    eval_steps=10,                          # 10 step마다 시험 봄
    generation_max_length=256,              # 생성할 글자의 최대 길이 (256 토큰)

    # ─────── 저장 전략 ───────
    save_strategy="steps",                  # step 단위로 저장
    save_steps=10,                          # 10 step마다 모델 저장
    save_total_limit=5,                     # 최대 5개까지만 저장 (오래된 건 삭제)
                                            # → 디스크 용량 절약

    # ─────── 로그(기록) 관련 ───────
    logging_strategy="steps",               # step 단위로 기록
    logging_steps=10,                       # 10 step마다 화면에 진행상황 출력

    # ─────── 생성 모드 ───────
    predict_with_generate=True,             # 평가 시 실제로 텍스트 생성해서 비교
                                            # (단순 손실값이 아닌 진짜 받아쓰기 테스트)

    # ─────── "최고 모델" 판단 기준 ───────
    metric_for_best_model="cer",            # CER(글자 오류율) 기준으로 최고 모델 선정
    greater_is_better=False,                # CER은 낮을수록 좋음 → False
    load_best_model_at_end=True,            # 학습 끝나면 자동으로 최고 모델 로드

    # ─────── 기타 ───────
    remove_unused_columns=False,            # 데이터에서 안 쓰는 컬럼 제거 안 함
    label_names=["labels"],                 # 정답 컬럼 이름이 'labels'임을 명시
)


In [20]:
# ============================================================
# [19단계] 트레이너(학습 코치) 만들기
# ============================================================
# 이제까지 만든 모든 부품을 하나로 합쳐서 "학습 코치"를 만들어요.
# 트레이너가 알아서 학습을 진행시켜 줄 거예요.
#
# 트레이너에게 필요한 것:
#   - 학습 계획표 (training_args)
#   - 학습할 AI (model)
#   - 교과서 (train_dataset)
#   - 모의고사 (eval_dataset)
#   - 데이터 정리 도구 (data_collator)
#   - 채점 기준 (compute_metrics)
#   - 처리 도구 (tokenizer)
# ============================================================

trainer = Seq2SeqTrainer(
    args=training_args,                # 18단계에서 만든 학습 옵션
    model=model,                       # 11단계에서 만든 LoRA 적용 모델
    train_dataset=datasets["train"],   # 학습용 데이터 (3,400개)
    eval_dataset=datasets["test"],     # 평가용 데이터 (340개)
    data_collator=data_collator,       # 13단계에서 만든 데이터 묶음 도구
    compute_metrics=compute_metrics,   # 15단계의 점수 계산 함수
    tokenizer=processor.feature_extractor,  # 처리기 (음성→숫자)
                                            # ※ 최신 버전에선 processing_class를 권장
)


/tmp/ipykernel_4194249/3662170775.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [21]:
# ============================================================
# [20단계] 드디어 학습 시작!
# ============================================================
# 단 한 줄로 모든 학습이 시작돼요.
# 이제까지 19단계 동안 준비한 모든 것이 이 한 줄에서 작동해요.
#
# 학습 중 화면에서 볼 수 있는 것:
#   - Loss(손실값): 낮아질수록 AI가 잘 배우고 있는 것
#   - WER, CER: 평가 점수 (낮을수록 좋음)
#   - 진행 막대: 100 step 중 몇 번째인지
#   - 예상 남은 시간
#
# 학습 결과 예시 (실행 후 출력):
#   - global_step=100: 100번 학습 완료
#   - training_loss=0.087: 손실값 0.087 (꽤 낮은 편 = 잘 배움)
#   - train_runtime=2332초 (약 39분 소요)
#   - epoch=1.86: 전체 데이터셋을 약 1.86바퀴 본 셈
#
# [주의] 학습은 GPU 환경에서 수십 분~수 시간 걸려요!
# ============================================================

trainer.train()


Step,Training Loss,Validation Loss,Wer,Cer,Char Accuracy
10,0.634000,0.323807,26.176471,2.534609,95.265711
20,0.145200,0.055098,11.176471,0.926303,98.797976
30,0.036800,0.018833,3.823529,0.356270,99.272459
40,0.016300,0.011252,2.058824,0.203583,99.620413
50,0.012200,0.008284,0.735294,0.050896,99.947280
60,0.007800,0.006595,0.441176,0.030537,99.968368
70,0.006000,0.005861,0.294118,0.020358,99.978912
80,0.005900,0.005431,0.147059,0.010179,99.989456
90,0.006000,0.005169,0.294118,0.020358,99.978912
100,0.005800,0.005071,0.294118,0.020358,99.978912


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


TrainOutput(global_step=100, training_loss=0.08760792337357998, metrics={'train_runtime': 1747.3283, 'train_samples_per_second': 3.663, 'train_steps_per_second': 0.057, 'total_flos': 1.100779249729536e+19, 'train_loss': 0.08760792337357998, 'epoch': 1.8598130841121496})